In [34]:
!pip install -q langchain langchain-groq

import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Day 3 setup successful!")

Day 3 setup successful!


In [35]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS: Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
       Orders need 24 hours' notice. All cakes have eggless versions.
RULES: Never promise a delivery time or place. If it isn't in FACTS,
       say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Yes, Ravi! All our cakes—plum, chocolate, and cupcakes—are available eggless. Let me know what you’d like and the quantity, and I’ll calculate the total for you.
Your name is Ravi.


In [36]:
print(ask("Can I order an eggless plum cake?"))

Sure thing, Ravi! An eggless plum cake is Rs 500. Please let me know how many you’d like, and we’ll confirm the order (24‑hour notice required). Let me know the quantity!


In [37]:
import csv

tests = [
    ["What is the price of plum cake?", "500"],
    ["What is the price of chocolate cake?", "650"],
    ["How much is one cupcake?", "80"],
    ["Are eggless cakes available?", "eggless"],
    ["How much notice is required?", "24"],
    ["Can you deliver in one hour?", "check with the owner"],
    ["What is the price of 5 cupcakes?", "400"],
    ["Can I order an eggless chocolate cake?", "eggless"],
    ["Do you sell vanilla cake?", "check with the owner"],
    ["Can I place an order for tomorrow?", "24"],
    ["What is the total for 2 plum cakes?", "1000"],
    ["What is the total for 3 chocolate cakes?", "1950"],
    ["Can you guarantee delivery at 6 PM?", "check with the owner"],
    ["Do you offer sugar-free cakes?", "check with the owner"],
    ["Can I get an eggless cupcake?", "eggless"],
    ["What is the total for 10 cupcakes?", "800"],
    ["Can I order a cake with no eggs?", "eggless"],
    ["Do you sell black forest cake?", "check with the owner"],
    ["Can you deliver today in 30 minutes?", "check with the owner"],
    ["What is the price of 2 chocolate cakes and 2 cupcakes?", "1460"]
]

with open("tests.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["question", "must_include"])
    writer.writerows(tests)

print("tests.csv created successfully!")
print("Total test cases:", len(tests))

tests.csv created successfully!
Total test cases: 20


In [38]:
import csv

def run_tests(path="tests.csv"):
    with open(path, newline="") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for i, row in enumerate(rows, 1):
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()

        if ok:
            passed += 1

        print(
            f"{'PASS' if ok else 'FAIL'} | Test {i}: "
            f"{row['question']}\nAnswer: {answer}\n"
        )

    print(f"FINAL SCORE: {passed}/{len(rows)}")
    print(f"PASS RATE: {passed / len(rows):.0%}")

run_tests()

PASS | Test 1: What is the price of plum cake?
Answer: The plum cake is Rs 500.

PASS | Test 2: What is the price of chocolate cake?
Answer: Chocolate cake costs Rs 650.

PASS | Test 3: How much is one cupcake?
Answer: One cupcake costs Rs 80.

PASS | Test 4: Are eggless cakes available?
Answer: Yes! All our cakes—plum, chocolate, and cupcakes—come in eggless versions. Let me know which you’d like, and I’ll add it to your order.

PASS | Test 5: How much notice is required?
Answer: We require a 24‑hour notice for all orders. Let me know what you’d like, and I’ll reserve it for you!

PASS | Test 6: Can you deliver in one hour?
Answer: I’m sorry, we need 24‑hour notice for orders. I’ll check with the owner.

PASS | Test 7: What is the price of 5 cupcakes?
Answer: 5 cupcakes cost Rs 400. Enjoy!

PASS | Test 8: Can I order an eggless chocolate cake?
Answer: Sure, we have an eggless chocolate cake for Rs 650. Please give us 24 hours’ notice to place your order.

PASS | Test 9: Do you sell va

In [39]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import csv

creative_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

creative_chain = prompt | creative_llm | StrOutputParser()

def run_temperature_tests():
    with open("tests.csv", newline="") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = creative_chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        passed += ok

        print("PASS" if ok else "FAIL", "|", row["question"])

    print(f"Score: {passed}/{len(rows)}")
    print(f"Pass rate: {passed / len(rows):.0%}")

print("RUN 1")
run_temperature_tests()

print("\nRUN 2")
run_temperature_tests()

RUN 1
PASS | What is the price of plum cake?
PASS | What is the price of chocolate cake?
PASS | How much is one cupcake?
PASS | Are eggless cakes available?
PASS | How much notice is required?
PASS | Can you deliver in one hour?
PASS | What is the price of 5 cupcakes?
PASS | Can I order an eggless chocolate cake?
PASS | Do you sell vanilla cake?
PASS | Can I place an order for tomorrow?
PASS | What is the total for 2 plum cakes?
FAIL | What is the total for 3 chocolate cakes?
PASS | Can you guarantee delivery at 6 PM?
PASS | Do you offer sugar-free cakes?
PASS | Can I get an eggless cupcake?
PASS | What is the total for 10 cupcakes?
PASS | Can I order a cake with no eggs?
PASS | Do you sell black forest cake?
PASS | Can you deliver today in 30 minutes?
FAIL | What is the price of 2 chocolate cakes and 2 cupcakes?
Score: 18/20
Pass rate: 90%

RUN 2
PASS | What is the price of plum cake?
PASS | What is the price of chocolate cake?
PASS | How much is one cupcake?
FAIL | Are eggless cakes 

In [40]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

def get_price(item: str, quantity: int = 1) -> int:
    """Calculate the total price of a menu item."""
    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not found in menu.")

    if quantity <= 0:
        raise ValueError("Quantity must be greater than zero.")

    return MENU[item] * quantity


def check_order_notice(hours_notice: int) -> bool:
    """Check whether an order meets the 24-hour notice requirement."""
    return hours_notice >= 24


print("Price:", get_price("cupcake", 6))
print("Order with 24 hours notice:", check_order_notice(24))
print("Order with 12 hours notice:", check_order_notice(12))

Price: 480
Order with 24 hours notice: True
Order with 12 hours notice: False


In [41]:
test_cases = [
    ("plum cake", 2),
    ("CHOCOLATE CAKE", 3),
    ("Cupcake", 10),
    ("vanilla cake", 1),
    ("cupcake", 0),
    ("cupcake", -2)
]

for item, quantity in test_cases:
    try:
        print(item, quantity, "=", get_price(item, quantity))
    except ValueError as e:
        print(item, quantity, "=", e)


print("\nNotice tests:")

for hours in [0, 12, 23, 24, 48]:
    print(hours, "hours =", check_order_notice(hours))

plum cake 2 = 1000
CHOCOLATE CAKE 3 = 1950
Cupcake 10 = 800
vanilla cake 1 = Item not found in menu.
cupcake 0 = Quantity must be greater than zero.
cupcake -2 = Quantity must be greater than zero.

Notice tests:
0 hours = False
12 hours = False
23 hours = False
24 hours = True
48 hours = True


In [42]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Do you make eggless chocolate cakes?"}
    ]
})

print(result["messages"][-1].content)

Yes! We offer an eggless chocolate cake for Rs 650. Just give us a 24‑hour notice and we’ll have it ready for you.


In [43]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

questions = [
    "Do you have eggless chocolate cake?",
    "What is the price of 5 cupcakes?",
    "Can you guarantee delivery within one hour?",
    "Can I order a vanilla cake?"
]

for question in questions:
    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    print("Customer:", question)
    print("Assistant:", result["messages"][-1].content)
    print("-" * 60)

Customer: Do you have eggless chocolate cake?
Assistant: Yes, we offer an eggless chocolate cake for Rs 650. Please place your order at least 24 hours in advance.
------------------------------------------------------------
Customer: What is the price of 5 cupcakes?
Assistant: 5 cupcakes cost Rs 400.
------------------------------------------------------------
Customer: Can you guarantee delivery within one hour?
Assistant: Sure! I'll check with the owner and get back to you.
------------------------------------------------------------
Customer: Can I order a vanilla cake?
Assistant: Sure! We don’t have vanilla listed, but I’ll check with the owner and let you know. Remember, we need 24‑hour notice for all orders.
------------------------------------------------------------


In [44]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("Improved SYSTEM prompt loaded successfully!")

Improved SYSTEM prompt loaded successfully!


In [45]:
run_tests()

PASS | Test 1: What is the price of plum cake?
Answer: The plum cake costs Rs 500.

PASS | Test 2: What is the price of chocolate cake?
Answer: Chocolate cake costs Rs 650. It’s also available in an eggless version if you’d like!

PASS | Test 3: How much is one cupcake?
Answer: One cupcake costs Rs 80. Let me know if you’d like to order!

PASS | Test 4: Are eggless cakes available?
Answer: Yes! All our cakes—plum, chocolate, and cupcakes—are available in eggless versions. Let me know which you’d like, and I can give you the price and details.

PASS | Test 5: How much notice is required?
Answer: You’ll need 24 hours’ notice before placing an order.

PASS | Test 6: Can you deliver in one hour?
Answer: I’ll check with the owner.

PASS | Test 7: What is the price of 5 cupcakes?
Answer: 5 cupcakes cost Rs 400.

PASS | Test 8: Can I order an eggless chocolate cake?
Answer: Sure! Eggless chocolate cake is Rs 650. Please give us 24‑hour notice. Let me know your order details.

PASS | Test 9: D

In [46]:
import csv

test_cases = [
    ("What is the price of plum cake?", "500"),
    ("What is the price of chocolate cake?", "650"),
    ("How much is one cupcake?", "80"),
    ("Do you have eggless cakes?", "eggless"),
    ("How much for 6 cupcakes?", "480"),
    ("How much for 2 plum cakes?", "1000"),
    ("Do orders require 24 hours notice?", "24"),
    ("Can you guarantee delivery in one hour?", "check with the owner"),
    ("Do you sell vanilla cake?", "check with the owner"),
    ("Can I order a chocolate cake today?", "24")
]

with open("tests.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["question", "must_include"])
    writer.writerows(test_cases)

print("tests.csv created successfully!")

tests.csv created successfully!


In [47]:
import pandas as pd

df = pd.read_csv("tests.csv")

print(df)
print("\nTotal test cases:", len(df))
print("Columns:", list(df.columns))

                                  question          must_include
0          What is the price of plum cake?                   500
1     What is the price of chocolate cake?                   650
2                 How much is one cupcake?                    80
3               Do you have eggless cakes?               eggless
4                 How much for 6 cupcakes?                   480
5               How much for 2 plum cakes?                  1000
6       Do orders require 24 hours notice?                    24
7  Can you guarantee delivery in one hour?  check with the owner
8                Do you sell vanilla cake?  check with the owner
9      Can I order a chocolate cake today?                    24

Total test cases: 10
Columns: ['question', 'must_include']


In [48]:
from google.colab import files
files.download("tests.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>